# Traffic Demand Prediction

The goal is to predict `demand` for each place and time. The score is R squared times 100.

My steps:
1. Look at the data and see what affects demand.
2. Build a few useful features.
3. Train three models and average them.
4. Save the predictions file.


In [1]:
import os, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from scipy.spatial import cKDTree
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostRegressor

# auto-detect dataset location (train.csv / test.csv)
CANDS = ['dataset/dataset/','dataset/','./','data/','../input/']
DATA = next((p for p in CANDS if os.path.exists(p+'train.csv')), 'dataset/dataset/')
train = pd.read_csv(DATA+'train.csv')
test  = pd.read_csv(DATA+'test.csv')
print('train', train.shape, '| test', test.shape, '| from', DATA)
train.head()

train (77299, 11) | test (41778, 10) | from dataset/dataset/


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


## Step 1 - Look at the data

A few things I noticed:
- Train has all of day 48 and the early hours of day 49. The test is the rest of day 49.
- Road type and number of lanes matter the most. Highways and 4-5 lane roads have much higher demand.
- Weather and temperature barely matter.


In [2]:
# temporal structure
def _slot(ts): h,m = ts.split(':'); return (int(h)*60+int(m))//15
for df in (train, test): df['_s'] = df['timestamp'].map(_slot)
print('train days:', sorted(train.day.unique()), '| test days:', sorted(test.day.unique()))
print('train day48 slots:', train[train.day==48]._s.nunique(),
      '| train day49 slots:', sorted(train[train.day==49]._s.unique()))
print('test day49 slots :', f"{test._s.min()}..{test._s.max()}")
print('\ndemand mean by RoadType:\n', train.groupby('RoadType')['demand'].mean())
print('\ndemand mean by NumberofLanes:\n', train.groupby('NumberofLanes')['demand'].mean())
print('\ncorr(demand, Temperature) =', round(train['demand'].corr(train['Temperature']),4),
      '(Weather/Temperature are ~noise)')
train.drop(columns='_s', inplace=True); test.drop(columns='_s', inplace=True)

train days: [np.int64(48), np.int64(49)] | test days: [np.int64(49)]
train day48 slots: 96 | train day49 slots: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
test day49 slots : 9..55

demand mean by RoadType:
 RoadType
Highway        0.610756
Residential    0.057209
Street         0.273164
Name: demand, dtype: float64

demand mean by NumberofLanes:
 NumberofLanes
1    0.088104
2    0.077488
3    0.077859
4    0.602882
5    0.607556
Name: demand, dtype: float64

corr(demand, Temperature) = 0.0031 (Weather/Temperature are ~noise)


## Step 2 - Build features

- Turn the geohash into latitude and longitude (it is a real location code).
- Get the time of day from the timestamp.
- Add yesterday's demand at the same place and time. This is the most useful feature.
- Day 49 is busier than day 48, so I add a simple ratio to adjust for it.
- Add the average demand for each location and its nearby locations.


In [3]:
B32 = "0123456789bcdefghjkmnpqrstuvwxyz"
def decode_geohash(gh):
    lat=[-90.,90.]; lon=[-180.,180.]; even=True
    for c in gh:
        cd=B32.index(c)
        for mask in (16,8,4,2,1):
            if even: mid=(lon[0]+lon[1])/2; lon[0 if cd&mask else 1]=mid
            else:    mid=(lat[0]+lat[1])/2; lat[0 if cd&mask else 1]=mid
            even=not even
    return (lat[0]+lat[1])/2, (lon[0]+lon[1])/2

allgh = pd.concat([train.geohash, test.geohash]).unique()
GLAT={}; GLON={}
for g in allgh:
    la,lo = decode_geohash(g); GLAT[g]=la; GLON[g]=lo

ROAD={'Residential':0,'Street':1,'Highway':2}; WEA={'Sunny':0,'Rainy':1,'Foggy':2,'Snowy':3}
def to_slot(ts): h,m=ts.split(':'); return (int(h)*60+int(m))//15

def base(df):
    d=df.copy()
    d['slot']=d['timestamp'].map(to_slot)
    d['sin']=np.sin(2*np.pi*d.slot/96); d['cos']=np.cos(2*np.pi*d.slot/96)
    d['lat']=d.geohash.map(GLAT); d['lon']=d.geohash.map(GLON)
    d['RoadType']=d.RoadType.map(ROAD).fillna(9).astype(int)
    d['Weather']=d.Weather.map(WEA).fillna(9).astype(int)
    d['LargeVehicles']=(d.LargeVehicles=='Allowed').astype(int)
    d['Landmarks']=(d.Landmarks=='Yes').astype(int)
    return d
train_b=base(train); test_b=base(test)

In [4]:
# --- lag = previous-day (day 48) demand at the SAME place and time ---
# The test is day 49. Every test row looks back to day 48.
# We build training lag the SAME way: day-49 rows read day 48;
# day-48 rows have no previous day here, so lag is left empty.
look48 = train_b[train_b.day==48].set_index(['geohash','slot'])['demand'].to_dict()
def addlag(d, test_mode=False):
    d=d.copy()
    is49 = np.ones(len(d),bool) if test_mode else (d.day.values==49)
    d['lag'] = [look48.get((g,s), np.nan) if i else np.nan
                for g,s,i in zip(d.geohash, d.slot, is49)]
    return d
train_b=addlag(train_b); test_b=addlag(test_b, test_mode=True)

In [5]:
# --- per-geohash day-to-day shift (day49/day48) from the known night slots 0-8 ---
NB=list(range(0,9)); g48={}; g49={}
d48n=train_b[(train_b.day==48)&(train_b.slot.isin(NB))]
d49n=train_b[(train_b.day==49)&(train_b.slot.isin(NB))]
for g,sub in d48n.groupby('geohash'): g48[g]=sub.set_index('slot')['demand'].to_dict()
for g,sub in d49n.groupby('geohash'): g49[g]=sub.set_index('slot')['demand'].to_dict()
GLOB_SHIFT=d49n['demand'].sum()/d48n['demand'].sum()
def _shift(g, excl=None):
    a=g49.get(g,{}); b=g48.get(g,{}); common=set(a)&set(b)
    if excl is not None: common=common-{excl}
    if not common: return GLOB_SHIFT
    num=sum(a[s] for s in common); den=sum(b[s] for s in common)
    return num/den if den>1e-9 else GLOB_SHIFT
ds=np.full(len(train_b), np.nan)
for i,(g,s,dy) in enumerate(zip(train_b.geohash.values,train_b.slot.values,train_b.day.values)):
    if dy==49: ds[i]=_shift(g, excl=s)            # leave-one-out on train (no leakage)
train_b['dayshift']=np.clip(ds,0.2,5.0)
test_b['dayshift']=np.clip([_shift(g) for g in test_b.geohash],0.2,5.0)
print('global day49/day48 shift =', round(GLOB_SHIFT,3))

global day49/day48 shift = 1.722


In [6]:
# --- target encoding (OOF) + spatial neighbour helpers ---
def te_fit(tr,col,a=10.):
    g=tr['demand'].mean(); st=tr.groupby(col)['demand'].agg(['sum','count'])
    return (st['sum']+g*a)/(st['count']+a), g
def te_apply(df,col,enc,g): return df[col].map(enc).fillna(g).values
def te_oof(tr,col,a=10.,n=5,seed=42):
    oof=np.full(len(tr),np.nan); kf=KFold(n,shuffle=True,random_state=seed)
    for fi,vi in kf.split(tr):
        enc,g=te_fit(tr.iloc[fi],col,a); oof[vi]=te_apply(tr.iloc[vi],col,enc,g)
    return oof
ghs=list(allgh); coords=np.array([[GLAT[g],GLON[g]] for g in ghs])
nbr_idx=cKDTree(coords).query(coords,k=9)[1]; GH2I={g:i for i,g in enumerate(ghs)}
def nbr_te(df,gmap,glob):
    vals=np.array([gmap.get(g,glob) for g in ghs])
    return np.array([vals[nbr_idx[GH2I[g]][1:]].mean() for g in df.geohash])

## Step 3 - Check the score

There are no daytime answers for day 49, so I hold out the last few night hours that I do have and predict them from only day 48 plus the earlier night hours - exactly the way the real test works. This gives an honest score of about 0.94 (R squared). The leaderboard hours are full daytime and harder, so the real score may be a little lower.

In [7]:
FEATS_A = ['slot','sin','cos','lat','lon','RoadType','NumberofLanes','LargeVehicles',
           'Landmarks','Weather','Temperature','geohash_te','nbr_te','lag','dayshift']
FEATS_B = [f for f in FEATS_A if f != 'dayshift']     # twin without day-shift (hedge)
CATL    = ['RoadType','Weather']

def fit_lgb(Xt, yt, Xv, yv, Xe, feats, seed=1):
    cats = [c for c in CATL if c in feats]
    dtr  = lgb.Dataset(Xt[feats], yt, categorical_feature=cats)
    dv   = lgb.Dataset(Xv[feats], yv, reference=dtr, categorical_feature=cats)
    p = dict(objective='regression', metric='l2', learning_rate=0.03, num_leaves=127,
             min_data_in_leaf=30, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
             lambda_l1=0.1, lambda_l2=0.5, verbose=-1, n_jobs=-1, seed=seed)
    m = lgb.train(p, dtr, 3000, valid_sets=[dv],
                  callbacks=[lgb.early_stopping(120, verbose=False)])
    return np.clip(m.predict(Xv[feats]),0,1), np.clip(m.predict(Xe[feats]),0,1)

# --- honest forecast check: hold out LATER night slots, rebuild day-shift from EARLY ones only ---
def honest_split(hist, hold):
    tr = train_b[(train_b.day==48) | ((train_b.day==49) & (train_b.slot.isin(hist)))].copy()
    va = train_b[(train_b.day==49) & (train_b.slot.isin(hold))].copy()
    a={};  b={}
    for gg,s in tr[tr.day==49].groupby('geohash'): a[gg]=s.set_index('slot')['demand'].to_dict()
    for gg,s in tr[(tr.day==48)&(tr.slot.isin(hist))].groupby('geohash'): b[gg]=s.set_index('slot')['demand'].to_dict()
    gsh = sum(v for d in a.values() for v in d.values()) / max(sum(v for d in b.values() for v in d.values()),1e-9)
    def sh(gg):
        x=a.get(gg,{}); y=b.get(gg,{}); c=set(x)&set(y); den=sum(y[s] for s in c)
        return (sum(x[s] for s in c)/den) if c and den>1e-9 else gsh
    for d in (tr,va): d['dayshift']=np.clip([sh(gg) for gg in d.geohash],0.2,5.0)
    enc,g = te_fit(tr,'geohash'); gm=enc.to_dict()
    for d in (tr,va): d['geohash_te']=te_apply(d,'geohash',enc,g); d['nbr_te']=nbr_te(d,gm,g)
    return tr, va

tr_h, va_h = honest_split(list(range(0,6)), [6,7,8])
pvA, _ = fit_lgb(tr_h, tr_h['demand'], va_h, va_h['demand'], va_h, FEATS_A)
pvB, _ = fit_lgb(tr_h, tr_h['demand'], va_h, va_h['demand'], va_h, FEATS_B)
print("honest forecast R2 =", round(r2_score(va_h['demand'], 0.5*pvA+0.5*pvB), 4))

honest forecast R2 = 0.9397


## Step 4 - Train the models

I train LightGBM twice - once with the day-shift feature and once without it - then average the two. The day-shift is measured from night hours so mixing in a model that ignores it stops the daytime predictions from being pushed too far in one direction.

In [8]:
full = train_b.reset_index(drop=True).copy()
enc_f, g_f = te_fit(full,'geohash'); gmap_f=enc_f.to_dict()
test_b['geohash_te'] = te_apply(test_b,'geohash',enc_f,g_f)
test_b['nbr_te']     = nbr_te(test_b, gmap_f, g_f)

kf = KFold(5, shuffle=True, random_state=1)
tA = np.zeros(len(test_b)); tB = np.zeros(len(test_b))
for fold,(fi,vi) in enumerate(kf.split(full)):
    Xt=full.iloc[fi].copy(); Xv=full.iloc[vi].copy()
    Xt['geohash_te'] = te_oof(Xt,'geohash')
    enc,g = te_fit(Xt,'geohash'); gmap=enc.to_dict()
    Xv['geohash_te'] = te_apply(Xv,'geohash',enc,g)
    Xt['nbr_te'] = nbr_te(Xt,gmap,g); Xv['nbr_te'] = nbr_te(Xv,gmap,g)
    yt, yv = Xt['demand'], Xv['demand']
    _,ptA = fit_lgb(Xt,yt,Xv,yv,test_b,FEATS_A,seed=100+fold); tA += ptA/5
    _,ptB = fit_lgb(Xt,yt,Xv,yv,test_b,FEATS_B,seed=200+fold); tB += ptB/5
    print('fold',fold,'done')

fold 0 done


fold 1 done


fold 2 done


fold 3 done


fold 4 done


## Step 5 - Results and save the predictions


In [9]:
ens_test   = np.clip(0.5*tA + 0.5*tB, 0, 1)
submission = pd.DataFrame({'Index': test['Index'].values, 'demand': ens_test})
submission.to_csv('submission.csv', index=False)
print('saved submission.csv', submission.shape)
print('prediction mean =', round(ens_test.mean(),4))
submission.head()

saved submission.csv (41778, 2)
prediction mean = 0.1269


,Index,demand
0,0,0.051643
1,1,0.028550
2,2,0.045128
3,3,0.036308
4,4,0.047312
